### **How Images Work**

Include an image block alongside a text block in your user message. Images can be sent as base64 or a URL.

### **Limits to Know**

*   Max 100 images per request
    
*   Max 5MB per image
    
*   Single image: 8000px max
    
*   Multiple images: 2000px max each
    
*   Token cost: (width × height) / 750
    

### **Getting Good Results**

Same rules as text prompting — simple questions give poor results. Give Claude a methodology, break tasks into steps, and use examples when accuracy matters.

In [7]:
import anthropic
import base64
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()
model = "claude-sonnet-4-6"

In [8]:
def load_image(path):
    with open(path, "rb") as f:
        return base64.standard_b64encode(f.read()).decode("utf-8")

def image_block(path, media_type="image/png"):
    return {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": media_type,
            "data": load_image(path)
        }
    }

def text_block(text):
    return {"type": "text", "text": text}

In [9]:
def ask_about_image(image_path, question, media_type="image/png"):
    messages = [{
        "role": "user",
        "content": [image_block(image_path, media_type), text_block(question)]
    }]
    response = client.messages.create(
        model=model,
        max_tokens=1024,
        messages=messages
    )
    return response.content[0].text

In [10]:
simple_prompt = "How many marbles are in this image?"

detailed_prompt = """Analyze this image of marbles and determine the exact count using this methodology:
1. Identify each unique marble one at a time. Assign each a number as you identify it.
2. Verify your result by counting with a different method — start from the bottom-left corner and work row by row, left to right.

What is the exact, verified number of marbles?"""

In [11]:
image_path = "./marbles.png"
media_type = "image/webp"  # file has .png extension but is actually WebP

print("Simple prompt result:")
print(ask_about_image(image_path, simple_prompt, media_type))

print("\nDetailed prompt result:")
print(ask_about_image(image_path, detailed_prompt, media_type))

Simple prompt result:
Looking at the image carefully, I can count **9 marbles** held in the palm of the hand. They are clear glass marbles with colorful swirls inside, in various colors including **blue, red/orange, green, yellow, and teal**.

Detailed prompt result:
## Step 1: Individual Identification

Let me identify each marble one at a time:

1. **Marble 1** – Blue marble, top-left area
2. **Marble 2** – Blue marble, top-right area
3. **Marble 3** – Green marble, middle-left
4. **Marble 4** – Red/orange marble, center
5. **Marble 5** – Red/orange marble, center-right
6. **Marble 6** – Yellow-green marble, lower-center
7. **Marble 7** – Teal/light blue marble, bottom-center
8. **Marble 8** – Clear/white marble, bottom-right
9. **Marble 9** – Olive/dark green marble, middle area

**Count: 9 marbles**

---

## Step 2: Verification (Bottom-left to top-right, row by row)

- **Bottom row:** teal, clear/white = **2**
- **Middle row:** green, yellow-green, red/orange, red/orange, olive = 